In [1]:
import os
os.environ["GEMINI_API_KEY"] = "USE_YOUR_API_KEY"

In [2]:
!pip install -qU langchain-google-genai langchain-chroma

In [3]:
!pip install -qU langchain-huggingface sentence-transformers

In [4]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# # Create LangChain documents for IPL players

In [5]:
from langchain_core.documents import Document

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [7]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(
        model="text-embedding-001"
    ),
    persist_directory="chroma_db",
    collection_name="sample",
)

**By HuggingFace Because GeminiAPI not working properly.**

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

# Free aur reliable HuggingFace embedding model initialize karein
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Chroma vector store create karein
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db",
    collection_name="sample",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Add Document into Vector Store

In [9]:
vector_store.add_documents(docs)

['d996c275-1b2b-4019-99ab-c020b5983a9d',
 '546fc2c7-7b4a-430e-a65d-62baa7bec4da',
 'ba47a862-dd63-46d9-bc68-b7d465d24fc5',
 'c097c978-9255-44d2-9f00-3f54ed195d34',
 '4431e17c-1947-4ed9-b699-98c4f3579677']

# View documents

In [10]:
vector_store.get(
    include = [
        "embeddings",
        "documents",
        "metadatas"
    ]
)

{'ids': ['66d3493d-0a1c-4d87-aa91-dccfd814567f',
  '60459dba-1f7d-4215-91de-a7a7601e5a19',
  'a9e19388-a132-4710-a1b9-65a9fdb6cae4',
  'f92a197f-285e-44ee-9e30-a6f1ad90b07d',
  '4fe2c2a7-d2da-4e83-b3a2-cad16eab704f',
  'd996c275-1b2b-4019-99ab-c020b5983a9d',
  '546fc2c7-7b4a-430e-a65d-62baa7bec4da',
  'ba47a862-dd63-46d9-bc68-b7d465d24fc5',
  'c097c978-9255-44d2-9f00-3f54ed195d34',
  '4431e17c-1947-4ed9-b699-98c4f3579677'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        ...,
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, 

# Search documents

In [11]:
vector_store.similarity_search(
    query = 'Who among these are a bowler?',
    k=2
)

[Document(id='f92a197f-285e-44ee-9e30-a6f1ad90b07d', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='c097c978-9255-44d2-9f00-3f54ed195d34', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

**# search with similarity score**

In [12]:
vector_store.similarity_search_with_score(
    query = 'Who among these are a bowler?',
    k=2
)

[(Document(id='f92a197f-285e-44ee-9e30-a6f1ad90b07d', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866),
 (Document(id='c097c978-9255-44d2-9f00-3f54ed195d34', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866)]

# Meta-Data Filtering

In [13]:
vector_store.similarity_search_with_score(
    query = 'Who among these are a bowler?',
    k=2,
    filter = {
        "team" : "Chennai Super Kings"
    }
)

[(Document(id='4fe2c2a7-d2da-4e83-b3a2-cad16eab704f', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1851273775100708),
 (Document(id='4431e17c-1947-4ed9-b699-98c4f3579677', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.1851273775100708)]

#Update documents

In [14]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

# Isme jo document_id hai wo Virat Kohli wale document ka doucment_id hai jise upar wali cell se lekr yaha use kiya hai
vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

# View documents

In [15]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['66d3493d-0a1c-4d87-aa91-dccfd814567f',
  '60459dba-1f7d-4215-91de-a7a7601e5a19',
  'a9e19388-a132-4710-a1b9-65a9fdb6cae4',
  'f92a197f-285e-44ee-9e30-a6f1ad90b07d',
  '4fe2c2a7-d2da-4e83-b3a2-cad16eab704f',
  'd996c275-1b2b-4019-99ab-c020b5983a9d',
  '546fc2c7-7b4a-430e-a65d-62baa7bec4da',
  'ba47a862-dd63-46d9-bc68-b7d465d24fc5',
  'c097c978-9255-44d2-9f00-3f54ed195d34',
  '4431e17c-1947-4ed9-b699-98c4f3579677'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        ...,
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, 

# Delete document

In [16]:
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['66d3493d-0a1c-4d87-aa91-dccfd814567f',
  '60459dba-1f7d-4215-91de-a7a7601e5a19',
  'a9e19388-a132-4710-a1b9-65a9fdb6cae4',
  'f92a197f-285e-44ee-9e30-a6f1ad90b07d',
  '4fe2c2a7-d2da-4e83-b3a2-cad16eab704f',
  'd996c275-1b2b-4019-99ab-c020b5983a9d',
  '546fc2c7-7b4a-430e-a65d-62baa7bec4da',
  'ba47a862-dd63-46d9-bc68-b7d465d24fc5',
  'c097c978-9255-44d2-9f00-3f54ed195d34',
  '4431e17c-1947-4ed9-b699-98c4f3579677'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        ...,
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, 